# CSE428 Spring 26 Project — Oxford-IIIT Pet (Segmentation + Classification)

**Name:** ____________________________  
**Student ID:** ______________________  

This notebook is structured to match the **presentation requirements** from the guideline PDF:

- Dataset exploration (boundary→foreground, 3×3 overlay grid)
- Base U-Net + classifier head (joint training)
- Attention U-Net + classifier head (joint training)
- Training curves (loss + metrics per epoch)
- Result summaries on **train/val/test**
- Demo visualization in the exact required 3-panel format

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import matplotlib.pyplot as plt

from cse428_pets.splits import make_splits
from cse428_pets.viz import plot_random_overlay_grid

In [ ]:
# Build deterministic splits (train/val from trainval.txt, plus test.txt)
splits = make_splits(repo_root=REPO_ROOT, val_ratio=0.2, seed=42)
print('train:', len(splits.train), 'val:', len(splits.val), 'test:', len(splits.test))

## Dataset exploration (3×3 overlay grid)
This cell shows 9 random samples with **true mask overlays** (boundary pixels included as foreground).

In [ ]:
plot_random_overlay_grid(samples=[*splits.train, *splits.val], label_info=splits.label_info, seed=0)

## Train (Base U-Net + classifier head)
Training runs are saved under `runs/<run_name>/` with `history.csv`, `best.pt`, and `config.json`.

Edit `run_name_unet` to avoid overwriting a previous run.

In [ ]:
import subprocess

run_name_unet = 'unet_run'  # change this if needed
cmd = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'train.py'),
    '--model', 'unet',
    '--epochs', '20',
    '--batch-size', '8',
    '--image-size', '256',
    '--run-name', run_name_unet,
]
print(' '.join(cmd))
# Uncomment to train:
# subprocess.run(cmd, check=True)

## Train (Attention U-Net + classifier head)
Edit `run_name_attn` to avoid overwriting a previous run.

In [ ]:
run_name_attn = 'attunet_run'  # change this if needed
cmd = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'train.py'),
    '--model', 'attunet',
    '--epochs', '20',
    '--batch-size', '8',
    '--image-size', '256',
    '--run-name', run_name_attn,
]
print(' '.join(cmd))
# Uncomment to train:
# subprocess.run(cmd, check=True)

## Training curves (loss + per-epoch validation metrics)
This plots `history.csv` produced by the training script.

In [ ]:
import csv

def load_history(run_name: str):
    path = REPO_ROOT / 'runs' / run_name / 'history.csv'
    rows = []
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append({k: float(v) if k != 'epoch' else int(v) for k, v in r.items()})
    return rows

def plot_history(rows, title: str):
    epochs = [r['epoch'] for r in rows]
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(epochs, [r['train_loss'] for r in rows], label='train')
    plt.plot(epochs, [r['val_loss'] for r in rows], label='val')
    plt.title('Total loss')
    plt.xlabel('epoch')
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(epochs, [r['val_miou'] for r in rows], label='val mIoU')
    plt.plot(epochs, [r['val_dice_fg'] for r in rows], label='val dice')
    plt.title('Segmentation metrics')
    plt.xlabel('epoch')
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(epochs, [r['val_f1'] for r in rows], label='val F1')
    plt.plot(epochs, [r['val_acc'] for r in rows], label='val accuracy')
    plt.title('Classification metrics')
    plt.xlabel('epoch')
    plt.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Example:
# rows = load_history(run_name_unet)
# plot_history(rows, title=run_name_unet)

## Result summaries (train/val/test)
Runs `scripts/evaluate.py` to generate `metrics.json`, then prints the required metrics for each split.

In [ ]:
def run_evaluate(run_name: str, weights: str = 'best.pt') -> None:
    cmd = [
        sys.executable,
        str(REPO_ROOT / 'scripts' / 'evaluate.py'),
        '--run-dir', str(REPO_ROOT / 'runs' / run_name),
        '--weights', weights,
    ]
    print(' '.join(cmd))
    # Uncomment to run evaluation:
    # subprocess.run(cmd, check=True)


def show_metrics(run_name: str) -> None:
    mpath = REPO_ROOT / 'runs' / run_name / 'metrics.json'
    metrics = json.loads(mpath.read_text(encoding='utf-8'))

    for split in ['train', 'val', 'test']:
        seg = metrics[split]['segmentation']
        cls = metrics[split]['classification']
        print(f"\n[{split}]\n"
              f"  Segmentation: mIoU={seg['miou']:.4f}, dice={seg['dice_fg']:.4f}, pixelAcc={seg['pixel_accuracy']:.4f}\n"
              f"  Classification: acc={cls['accuracy']:.4f}, precision={cls['precision']:.4f}, recall={cls['recall']:.4f}, F1={cls['f1']:.4f}")


# Example:
# run_evaluate(run_name_unet)
# show_metrics(run_name_unet)

## Required 3-panel prediction visualization
This reproduces the exact demo format from the guideline PDF: original image, true overlay, predicted overlay, plus original/predicted class and IoU.

In [ ]:
import random
import numpy as np
import torch
from PIL import Image
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

from cse428_pets.dataset import prepare_mask, species_name
from cse428_pets.models import UNetMultiTask
from cse428_pets.viz import compute_iou_binary, plot_prediction_triplet


def show_prediction(run_name: str, split: str = 'test', index: int = -1, weights: str = 'best.pt') -> None:
    run_dir = REPO_ROOT / 'runs' / run_name
    cfg = json.loads((run_dir / 'config.json').read_text(encoding='utf-8'))
    use_attention = cfg.get('model', 'unet') == 'attunet'
    image_size = int(cfg.get('image_size', 256))
    base_channels = int(cfg.get('base_channels', 64))
    splits = make_splits(repo_root=REPO_ROOT, val_ratio=float(cfg.get('val_ratio', 0.2)), seed=int(cfg.get('seed', 42)))

    samples = {'train': splits.train, 'val': splits.val, 'test': splits.test}[split]
    if index < 0:
        index = random.randint(0, len(samples) - 1)
    s = samples[index]

    model = UNetMultiTask(num_classes=37, use_attention=use_attention, base_channels=base_channels)
    ckpt = torch.load(run_dir / weights, map_location='cpu')
    model.load_state_dict(ckpt['model'])
    model.eval()

    img_pil = Image.open(s.image_path).convert('RGB')
    trimap = np.array(Image.open(s.mask_path), dtype=np.uint8)
    mask_true01 = prepare_mask(trimap, mode='binary')

    img_pil = TF.resize(img_pil, size=[image_size, image_size], interpolation=InterpolationMode.BILINEAR)
    mask_pil = Image.fromarray((mask_true01 * 255).astype(np.uint8), mode='L')
    mask_pil = TF.resize(mask_pil, size=[image_size, image_size], interpolation=InterpolationMode.NEAREST)
    mask_true01 = (np.array(mask_pil, dtype=np.uint8) > 127).astype(np.uint8)

    img_rgb = np.array(img_pil, dtype=np.uint8)

    x = TF.to_tensor(img_pil)
    x = TF.normalize(x, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    x = x.unsqueeze(0)

    with torch.no_grad():
        seg_logits, cls_logits = model(x)
        seg_prob = torch.sigmoid(seg_logits)[0, 0].numpy()
        mask_pred01 = (seg_prob >= 0.5).astype(np.uint8)
        pred_class = int(torch.argmax(cls_logits, dim=1).item())

    iou = compute_iou_binary(mask_true01, mask_pred01)

    def pretty(b: str) -> str:
        return b.replace('_', ' ')

    true_breed = pretty(splits.label_info.class_id_to_breed.get(s.class_id, str(s.class_id)))
    true_species = species_name(int(s.species_id))
    pred_breed = pretty(splits.label_info.class_id_to_breed.get(pred_class, str(pred_class)))
    pred_species = species_name(int(splits.label_info.class_id_to_species.get(pred_class, 0)))

    plot_prediction_triplet(
        image_rgb=img_rgb,
        mask_true01=mask_true01,
        mask_pred01=mask_pred01,
        true_label=f'{true_breed} ({true_species})',
        pred_label=f'{pred_breed} ({pred_species})',
        iou=iou,
    )


# Example:
# show_prediction(run_name_unet, split='test', index=-1)